In [2]:
import os
import json
import multiprocessing as mp
from concurrent.futures import ProcessPoolExecutor, as_completed
from pathlib import Path

import numpy as np
import pandas as pd
import scanpy as sc
import lightgbm as lgb
import shap
from matplotlib import pyplot as plt
from scipy import stats
from sklearn.model_selection import train_test_split

In [3]:
# =========================
# input
# =========================
offline_path = [
    "/home/wangyucheng/projects/SeqEval_Bench_Data/DNase/GM12878/GM12878_offline.tsv", 
    "/home/wangyucheng/projects/SeqEval_Bench_Data/DNase/HepG2/HepG2_offline.tsv", 
    "/home/wangyucheng/projects/SeqEval_Bench_Data/DNase/K562/K562_offline.tsv",
    "/home/wangyucheng/projects/SeqEval_Bench_Data/EpisomalMPRA/A549/A549_offline.tsv", 
    "/home/wangyucheng/projects/SeqEval_Bench_Data/EpisomalMPRA/HCT116/HCT116_offline.tsv", 
    "/home/wangyucheng/projects/SeqEval_Bench_Data/EpisomalMPRA/HepG2/HepG2_offline.tsv", 
    "/home/wangyucheng/projects/SeqEval_Bench_Data/EpisomalMPRA/K562/K562_offline.tsv", 
    "/home/wangyucheng/projects/SeqEval_Bench_Data/EpisomalMPRA/SKNSH/SKNSH_offline.tsv", 
    "/home/wangyucheng/projects/SeqEval_Bench_Data/LentiMPRA/HepG2/HepG2_offline.tsv", 
    "/home/wangyucheng/projects/SeqEval_Bench_Data/LentiMPRA/K562/K562_offline.tsv", 
    "/home/wangyucheng/projects/SeqEval_Bench_Data/LentiMPRA/WTC11/WTC11_offline.tsv"
]

complete_path = [
    "/home/wangyucheng/projects/SeqEval_Bench_Data/DNase/GM12878/GM12878_complete.tsv", 
    "/home/wangyucheng/projects/SeqEval_Bench_Data/DNase/HepG2/HepG2_complete.tsv", 
    "/home/wangyucheng/projects/SeqEval_Bench_Data/DNase/K562/K562_complete.tsv", 
    "/home/wangyucheng/projects/SeqEval_Bench_Data/EpisomalMPRA/A549/A549_complete.tsv", 
    "/home/wangyucheng/projects/SeqEval_Bench_Data/EpisomalMPRA/HCT116/HCT116_complete.tsv", 
    "/home/wangyucheng/projects/SeqEval_Bench_Data/EpisomalMPRA/HepG2/HepG2_complete.tsv", 
    "/home/wangyucheng/projects/SeqEval_Bench_Data/EpisomalMPRA/K562/K562_complete.tsv", 
    "/home/wangyucheng/projects/SeqEval_Bench_Data/EpisomalMPRA/SKNSH/SKNSH_complete.tsv", 
    "/home/wangyucheng/projects/SeqEval_Bench_Data/LentiMPRA/HepG2/HepG2_complete.tsv", 
    "/home/wangyucheng/projects/SeqEval_Bench_Data/LentiMPRA/K562/K562_complete.tsv", 
    "/home/wangyucheng/projects/SeqEval_Bench_Data/LentiMPRA/WTC11/WTC11_complete.tsv"
]

target = "functional_score"
SAMPLE_NUM = 50000

LGB_PARAMS = {
    'objective': 'regression',
    'metric': 'mae',
    'boosting_type': 'gbdt',
    'num_leaves': 63,
    'learning_rate': 0.05,
    'feature_fraction': 0.7,
    'seed': 42,
}
TRAIN_ROUNDS = 100000
EARLY_STOPPING_ROUNDS = 500


In [4]:
# =========================
# main
# =========================
def tsv_to_h5ad_path(file_path, split):
    """Map a raw tsv path to the site-count h5ad produced by prepare_tfbs_file."""
    p = Path(file_path)
    dataset, cell_type = p.parent.parent.name, p.parent.name
    tag = f"{dataset}_{cell_type}"
    h5ad_path = p.parent / "TACO" / split / f"{tag}__site_counts.h5ad"
    return tag, h5ad_path


def _process_one_file(file_path, split, target, sample_num, lgb_params,
                      train_rounds, early_stopping_rounds, seed, num_threads):
    """Train LightGBM on one dataset and infer TFBS rewards via SHAP.

    Returns (file_path, success, message).
    """
    try:
        tag, h5ad_path = tsv_to_h5ad_path(file_path, split)

        # save_dir: <input file's directory> / TACO / offline or complete
        save_dir = h5ad_path.parent
        os.makedirs(save_dir, exist_ok=True)
        model_out = save_dir / f"{tag}__site_counts_lgb_best.txt"
        json_out = save_dir / f"{tag}__site_counts_lgb_best.json"
        fig_out = save_dir / f"{tag}__tfbs_shap_barplot.pdf"

        data = sc.read_h5ad(h5ad_path)
        if hasattr(data.X, "toarray"):  # densify sparse matrices
            data.X = data.X.toarray()
        if data.n_obs > sample_num:
            rng = np.random.default_rng(seed)
            idx = rng.choice(data.n_obs, sample_num, replace=False)
            data = data[idx].copy()

        train_adata, temp_adata = train_test_split(
            data, test_size=0.2, random_state=seed, shuffle=True
        )
        val_adata, test_adata = train_test_split(
            temp_adata, test_size=0.5, random_state=seed, shuffle=True
        )

        # Extract features (X) and labels (y) for training, validation, and testing
        X_train, y_train = train_adata.X, train_adata.obs[target].values
        X_valid, y_valid = val_adata.X, val_adata.obs[target].values
        X_test, y_test = test_adata.X, test_adata.obs[target].values

        # Train LightGBM (threads capped per worker to avoid oversubscription)
        params = dict(lgb_params)
        params["num_threads"] = num_threads
        train_data = lgb.Dataset(X_train, label=y_train)
        valid_data = lgb.Dataset(X_valid, label=y_valid, reference=train_data)
        model = lgb.train(
            params,
            train_data,
            num_boost_round=train_rounds,
            valid_sets=[train_data, valid_data],
            valid_names=["train", "valid"],
            callbacks=[lgb.early_stopping(stopping_rounds=early_stopping_rounds, verbose=False)],
        )
        model.save_model(str(model_out))

        # SHAP analysis on the held-out test set
        feature_names = data.var.index.tolist()
        booster = lgb.Booster(model_file=str(model_out))
        explainer = shap.TreeExplainer(booster)
        shap_values = explainer(X_test)
        adjusted_shap_values_matrix = shap_values.values * (X_test != 0)

        mean_shap_values, max_shap_values, min_shap_values = [], [], []
        t_stats, p_values = [], []
        for i in range(len(feature_names)):
            non_zero_shap_values = adjusted_shap_values_matrix[X_test[:, i] > 0, i]
            if len(non_zero_shap_values) > 0:
                mean_shap_values.append(np.mean(non_zero_shap_values))
                max_shap_values.append(np.max(non_zero_shap_values))
                min_shap_values.append(np.min(non_zero_shap_values))
            else:
                mean_shap_values.append(np.nan)
                max_shap_values.append(np.nan)
                min_shap_values.append(np.nan)
            if len(non_zero_shap_values) > 1:
                t_stat, p_value = stats.ttest_1samp(non_zero_shap_values, 0)
            else:
                t_stat, p_value = np.nan, np.nan
            t_stats.append(t_stat)
            p_values.append(p_value)

        results_df = pd.DataFrame({
            "TFBS": feature_names,
            "Mean_SHAP": mean_shap_values,
            "Max_SHAP": max_shap_values,
            "Min_SHAP": min_shap_values,
            "T_stat": t_stats,
            "P_value": p_values,
        })
        significance_level = 0.05
        significant_results = results_df[results_df["P_value"] < significance_level]
        promoting_tfbs = significant_results[significant_results["Mean_SHAP"] > 0]
        inhibiting_tfbs = significant_results[significant_results["Mean_SHAP"] < 0]

        # Save the bar plot (plt.show() has no effect inside worker processes)
        plt.figure(figsize=(10, 6))
        if not promoting_tfbs.empty:
            plt.bar(promoting_tfbs["TFBS"], promoting_tfbs["Mean_SHAP"], color="green", label="Promoting TFBS")
        if not inhibiting_tfbs.empty:
            plt.bar(inhibiting_tfbs["TFBS"], inhibiting_tfbs["Mean_SHAP"], color="red", label="Inhibiting TFBS")
        plt.xticks(rotation=90)
        plt.ylabel("Mean SHAP Value")
        plt.title("Promoting and Inhibiting TFBS based on SHAP values (Significant Only)")
        plt.legend()
        plt.tight_layout()
        plt.savefig(fig_out)
        plt.close()

        result_dict = {}
        for _, row in results_df.iterrows():
            result_dict[row["TFBS"]] = row["Mean_SHAP"] if row["P_value"] < significance_level else 0
        with open(json_out, "w") as json_file:
            json.dump(result_dict, json_file, indent=4)

        return file_path, True, f"best_iter={model.best_iteration} -> {model_out} ; {json_out}"

    except Exception as e:
        return file_path, False, repr(e)


def run_tfbs_reward_inference(offline_paths=None, complete_paths=None,
                              target="functional_score", sample_num=50000,
                              lgb_params=None, train_rounds=100000,
                              early_stopping_rounds=500, seed=42, n_workers=1):
    """
    Train a LightGBM reward model on TFBS site counts and infer per-TFBS
    rewards via SHAP, for every offline / complete dataset.

    Input h5ad and all outputs live under <dataset dir>/TACO/offline or
    TACO/complete (created automatically if missing):
      - {tag}__site_counts_lgb_best.txt   (LightGBM model)
      - {tag}__site_counts_lgb_best.json  (TFBS -> reward dict)
      - {tag}__tfbs_shap_barplot.pdf      (SHAP bar plot)

    n_workers: number of parallel worker processes; 1 means sequential.
    """
    jobs = [(p, "offline") for p in (offline_paths or [])]
    jobs += [(p, "complete") for p in (complete_paths or [])]
    lgb_params = dict(lgb_params) if lgb_params else {}
    num_threads = max(1, (os.cpu_count() or 1) // max(1, n_workers))
    print(f"Total {len(jobs)} jobs, n_workers = {n_workers}, lgb threads per worker = {num_threads}")

    if n_workers <= 1:
        # Sequential execution (also useful for debugging)
        for i, (file_path, split) in enumerate(jobs, start=1):
            print("=" * 80)
            print(f"[{i}/{len(jobs)}] Processing ({split}): {file_path}")
            _, ok, msg = _process_one_file(file_path, split, target, sample_num, lgb_params,
                                           train_rounds, early_stopping_rounds, seed, num_threads)
            print(msg if ok else f"[ERROR] {msg}")
        return

    # Parallel execution: workers inherit notebook-defined functions via fork
    ctx = mp.get_context("fork")
    with ProcessPoolExecutor(max_workers=n_workers, mp_context=ctx) as executor:
        future_to_job = {
            executor.submit(_process_one_file, file_path, split, target, sample_num, lgb_params,
                            train_rounds, early_stopping_rounds, seed, num_threads): (file_path, split)
            for file_path, split in jobs
        }
        for done, future in enumerate(as_completed(future_to_job), start=1):
            file_path, split = future_to_job[future]
            _, ok, msg = future.result()
            status = "OK" if ok else "ERROR"
            print(f"[{done}/{len(jobs)}] [{status}] ({split}) {file_path}")
            print(f"    {msg}")




In [5]:
run_tfbs_reward_inference(
    offline_paths=offline_path,
    complete_paths=complete_path,
    target=target,
    sample_num=SAMPLE_NUM,
    lgb_params=LGB_PARAMS,
    train_rounds=TRAIN_ROUNDS,
    early_stopping_rounds=EARLY_STOPPING_ROUNDS,
    n_workers=4,
)


Total 22 jobs, n_workers = 4, lgb threads per worker = 32
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.019856 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 5841
[LightGBM] [Info] Number of data points in the train set: 21832, number of used features: 598
[LightGBM] [Info] Start training from score 1.432779
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.022197 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 6286
[LightGBM] [Info] Number of data points in the train set: 40000, number of used features: 598
[LightGBM] [Info] Start training from score 0.160952
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.019887 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, y